# FINANCE 384 Assignment 1 – Part A

## A.1 Data Setup and Prediction Design

The objective is to forecast each stock's next-month excess return using only information available at or before the monthly decision date \(t\).

The target is \(r^e_{i,t+1}\), constructed from `ret_excess_t` using a calendar-month join so that stocks that leave and later re-enter the S&P 500 are not incorrectly linked across gaps.

The base predictor set contains 18 admissible numeric characteristics together with Fama–French 49 industry membership. Identifiers and realised outcome variables are excluded.

Missing continuous characteristics are filled using the contemporaneous FF49 industry-month median, with the same-month market median as a fallback. Missingness indicators are retained. Continuous predictors are standardised using training-sample parameters, while FF49 industry membership is one-hot encoded.

The fixed chronological split is:

- Training: January 1990–December 2014
- Validation: January 2015–December 2018
- Test: January 2019–November 2022


In [1]:
# A.1.1 Imports and file locations

import pandas as pd
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

PANEL_FILE = "FINANCE384_assignmentA_development_panel.csv"
DICTIONARY_FILE = "FINANCE384_stock_month_data_dictionary.csv"

TARGET = "target_ret_excess_tp1"


In [2]:
# A.1.2 Load the development panel and data dictionary

panel = pd.read_csv(PANEL_FILE)
data_dictionary = pd.read_csv(DICTIONARY_FILE)

panel["date"] = pd.to_datetime(panel["date"])

print("Panel shape:", panel.shape)
print("Date range:",
      panel["date"].min().date(),
      "to",
      panel["date"].max().date())

print("Unique stocks:", panel["permno"].nunique())
print("Duplicate stock-month rows:",
      panel.duplicated(["permno", "date"]).sum())


Panel shape: (198298, 27)
Date range: 1990-01-31 to 2022-12-30
Unique stocks: 1252
Duplicate stock-month rows: 0


In [3]:
# A.1.3 Explicit predictor list

numeric_predictors = [
    "size",
    "bm",
    "mom12_2",
    "vol12",
    "beta60",
    "ivol60",
    "turnover",
    "dollar_volume",
    "amihud_illiq",
    "divyield",
    "gross_profit",
    "roe",
    "asset_growth",
    "leverage",
    "accruals",
    "mkt_12m",
    "mkt_vol_12m",
    "down_market",
]

continuous_predictors = [
    x for x in numeric_predictors
    if x != "down_market"
]

binary_predictors = ["down_market"]
categorical_predictors = ["ff49_code"]

identifier_columns = [
    "date",
    "permno",
    "ticker",
    "comnam",
    "gvkey",
]

print("Numeric predictors:", len(numeric_predictors))
print("Continuous predictors:", len(continuous_predictors))
print("Binary predictors:", binary_predictors)
print("Industry predictor:", categorical_predictors)


Numeric predictors: 18
Continuous predictors: 17
Binary predictors: ['down_market']
Industry predictor: ['ff49_code']


### Calendar-month target construction

The next-month excess-return target is attached using a calendar-month join rather than the next observed row. This prevents a stock that disappears and re-enters later from being incorrectly linked across a gap.


In [4]:
# A.1.4 Construct the next-calendar-month excess-return target

panel = panel.sort_values(
    ["permno", "date"]
).reset_index(drop=True)

panel["month"] = panel["date"].dt.to_period("M")

next_month_return = (
    panel[
        ["permno", "month", "ret_excess_t"]
    ]
    .rename(
        columns={
            "ret_excess_t": TARGET
        }
    )
    .assign(
        month=lambda df: df["month"] - 1
    )
)

panel = panel.merge(
    next_month_return,
    on=["permno", "month"],
    how="left",
    validate="one_to_one",
)

print("Rows in panel:", len(panel))
print("Rows with valid next-month target:",
      panel[TARGET].notna().sum())


Rows in panel: 198298
Rows with valid next-month target: 197016


### Missing-data treatment

Missing continuous characteristics are handled cross-sectionally using only information observable at the decision month:

1. same-month, same-FF49-industry median;
2. same-month market median as a fallback.

A missingness indicator is created before imputation so the models can retain information about whether the original characteristic was missing.


In [5]:
# A.1.5 Inspect missing characteristics before imputation

missing_summary = (
    panel[continuous_predictors]
    .isna()
    .sum()
    .to_frame("Missing observations")
)

missing_summary["Missing (%)"] = (
    100
    * missing_summary["Missing observations"]
    / len(panel)
)

missing_summary = (
    missing_summary
    .loc[missing_summary["Missing observations"] > 0]
    .sort_values(
        "Missing (%)",
        ascending=False
    )
)

missing_summary


,Missing observations,Missing (%)
beta60,17211,8.679361
ivol60,17211,8.679361
accruals,6615,3.335888
mom12_2,6023,3.037348
divyield,5927,2.988936
vol12,5268,2.656608
roe,4259,2.147778
bm,4242,2.139205
asset_growth,737,0.371663
gross_profit,515,0.259710


In [6]:
# A.1.6 Create missingness indicators before filling values

missing_characteristics = [
    col for col in continuous_predictors
    if panel[col].isna().any()
]

missing_indicator_columns = []

for col in missing_characteristics:
    indicator = f"{col}_was_missing"

    panel[indicator] = (
        panel[col]
        .isna()
        .astype(int)
    )

    missing_indicator_columns.append(indicator)

print("Characteristics with missing observations:")
print(missing_characteristics)

print("\nMissingness indicators created:")
print(missing_indicator_columns)


Characteristics with missing observations:
['bm', 'mom12_2', 'vol12', 'beta60', 'ivol60', 'divyield', 'gross_profit', 'roe', 'asset_growth', 'leverage', 'accruals']

Missingness indicators created:
['bm_was_missing', 'mom12_2_was_missing', 'vol12_was_missing', 'beta60_was_missing', 'ivol60_was_missing', 'divyield_was_missing', 'gross_profit_was_missing', 'roe_was_missing', 'asset_growth_was_missing', 'leverage_was_missing', 'accruals_was_missing']


In [7]:
# A.1.7 Cross-sectional missing-data treatment
#
# 1. Fill using the median for stocks in the same month and FF49 industry.
# 2. If unavailable, use the median across all stocks in the same month.

for col in continuous_predictors:

    industry_month_median = (
        panel
        .groupby(
            ["month", "ff49_code"]
        )[col]
        .transform("median")
    )

    market_month_median = (
        panel
        .groupby("month")[col]
        .transform("median")
    )

    panel[col] = (
        panel[col]
        .fillna(industry_month_median)
        .fillna(market_month_median)
    )


In [8]:
# A.1.8 Verify that missing predictor values have been resolved

remaining_missing = (
    panel[continuous_predictors]
    .isna()
    .sum()
)

remaining_missing = remaining_missing[
    remaining_missing > 0
]

if len(remaining_missing) == 0:
    print("All continuous predictor missing values were successfully filled.")
else:
    print("Remaining missing values:")
    print(remaining_missing)

print(
    "Missing down_market values:",
    panel["down_market"].isna().sum()
)

print(
    "Missing ff49_code values:",
    panel["ff49_code"].isna().sum()
)


All continuous predictor missing values were successfully filled.
Missing down_market values: 0
Missing ff49_code values: 0


In [9]:
# A.1.9 Keep observations with a valid next-month target

analysis = (
    panel
    .loc[panel[TARGET].notna()]
    .copy()
)

print("Usable stock-month observations:",
      len(analysis))

print("Forecast-date range:",
      analysis["date"].min().date(),
      "to",
      analysis["date"].max().date())


Usable stock-month observations: 197016
Forecast-date range: 1990-01-31 to 2022-11-30


### Fixed chronological samples

The assignment's prescribed train/validation/test periods are retained without modification.


In [10]:
# A.1.10 Fixed chronological samples

train = analysis.loc[
    (analysis["date"] >= "1990-01-01")
    & (analysis["date"] <= "2014-12-31")
].copy()

validation = analysis.loc[
    (analysis["date"] >= "2015-01-01")
    & (analysis["date"] <= "2018-12-31")
].copy()

test = analysis.loc[
    (analysis["date"] >= "2019-01-01")
    & (analysis["date"] <= "2022-11-30")
].copy()

sample_summary = pd.DataFrame({
    "Sample": [
        "Training",
        "Validation",
        "Test",
    ],
    "Start": [
        train["date"].min(),
        validation["date"].min(),
        test["date"].min(),
    ],
    "End": [
        train["date"].max(),
        validation["date"].max(),
        test["date"].max(),
    ],
    "Months": [
        train["month"].nunique(),
        validation["month"].nunique(),
        test["month"].nunique(),
    ],
    "Stock-month rows": [
        len(train),
        len(validation),
        len(test),
    ],
})

sample_summary


,Sample,Start,End,Months,Stock-month rows
0,Training,1990-01-31,2014-12-31,300,149334
1,Validation,2015-01-30,2018-12-31,48,24051
2,Test,2019-01-31,2022-11-30,47,23631


In [11]:
# A.1.11 Final base predictor information

raw_feature_columns = (
    continuous_predictors
    + binary_predictors
    + missing_indicator_columns
    + categorical_predictors
)

print(
    "Raw predictor columns supplied to preprocessing:",
    len(raw_feature_columns)
)

print("\nPredictors:")
for feature in raw_feature_columns:
    print("-", feature)


Raw predictor columns supplied to preprocessing: 30

Predictors:
- size
- bm
- mom12_2
- vol12
- beta60
- ivol60
- turnover
- dollar_volume
- amihud_illiq
- divyield
- gross_profit
- roe
- asset_growth
- leverage
- accruals
- mkt_12m
- mkt_vol_12m
- down_market
- bm_was_missing
- mom12_2_was_missing
- vol12_was_missing
- beta60_was_missing
- ivol60_was_missing
- divyield_was_missing
- gross_profit_was_missing
- roe_was_missing
- asset_growth_was_missing
- leverage_was_missing
- accruals_was_missing
- ff49_code


In [12]:
# A.1.12 Separate predictors and target

X_train_raw = train[raw_feature_columns].copy()
X_validation_raw = validation[raw_feature_columns].copy()
X_test_raw = test[raw_feature_columns].copy()

y_train = train[TARGET].to_numpy()
y_validation = validation[TARGET].to_numpy()
y_test = test[TARGET].to_numpy()

print("Training rows:", len(X_train_raw))
print("Validation rows:", len(X_validation_raw))
print("Test rows:", len(X_test_raw))


Training rows: 149334
Validation rows: 24051
Test rows: 23631


### Scaling and FF49 encoding

Continuous predictors are standardised using parameters estimated from the training sample only.

`down_market` and missingness indicators are passed through unchanged.

FF49 industry membership is one-hot encoded with one reference category omitted. Unknown categories in later samples are ignored safely.


In [13]:
# A.1.13 Preprocessing transformation

preprocessor = ColumnTransformer(
    transformers=[
        (
            "continuous",
            StandardScaler(),
            continuous_predictors,
        ),
        (
            "binary",
            "passthrough",
            binary_predictors
            + missing_indicator_columns,
        ),
        (
            "industry",
            OneHotEncoder(
                drop="first",
                handle_unknown="ignore",
                sparse_output=False,
            ),
            categorical_predictors,
        ),
    ],
    remainder="drop",
)


In [14]:
# A.1.14 Fit preprocessing using training data only

preprocessor.fit(X_train_raw)

X_train = preprocessor.transform(
    X_train_raw
)

X_validation = preprocessor.transform(
    X_validation_raw
)

X_test = preprocessor.transform(
    X_test_raw
)

print("Transformed training matrix:",
      X_train.shape)

print("Transformed validation matrix:",
      X_validation.shape)

print("Transformed test matrix:",
      X_test.shape)


Transformed training matrix: (149334, 75)
Transformed validation matrix: (24051, 75)
Transformed test matrix: (23631, 75)


In [15]:
# A.1.15 Retrieve transformed feature names for auditability

feature_names = (
    preprocessor
    .get_feature_names_out()
)

print(
    "Number of transformed predictors:",
    len(feature_names)
)

pd.Series(
    feature_names,
    name="Transformed predictor"
).head(30)


Number of transformed predictors: 75


0                     continuous__size
1                       continuous__bm
2                  continuous__mom12_2
3                    continuous__vol12
4                   continuous__beta60
5                   continuous__ivol60
6                 continuous__turnover
7            continuous__dollar_volume
8             continuous__amihud_illiq
9                 continuous__divyield
10            continuous__gross_profit
11                     continuous__roe
12            continuous__asset_growth
13                continuous__leverage
14                continuous__accruals
15                 continuous__mkt_12m
16             continuous__mkt_vol_12m
17                 binary__down_market
18              binary__bm_was_missing
19         binary__mom12_2_was_missing
20           binary__vol12_was_missing
21          binary__beta60_was_missing
22          binary__ivol60_was_missing
23        binary__divyield_was_missing
24    binary__gross_profit_was_missing
25             binary__ro

In [16]:
# A.1.16 Final A.1 audit

print("A.1 DATA AUDIT")
print("-" * 45)

print("Training observations:",
      len(train))

print("Validation observations:",
      len(validation))

print("Test observations:",
      len(test))

print(
    "\nMissing transformed training values:",
    np.isnan(X_train).sum()
)

print(
    "Missing transformed validation values:",
    np.isnan(X_validation).sum()
)

print(
    "Missing transformed test values:",
    np.isnan(X_test).sum()
)

print(
    "\nTraining target missing:",
    np.isnan(y_train).sum()
)

print(
    "Validation target missing:",
    np.isnan(y_validation).sum()
)

print(
    "Test target missing:",
    np.isnan(y_test).sum()
)


A.1 DATA AUDIT
---------------------------------------------
Training observations: 149334
Validation observations: 24051
Test observations: 23631

Missing transformed training values: 0
Missing transformed validation values: 0
Missing transformed test values: 0

Training target missing: 0
Validation target missing: 0
Test target missing: 0


## A.1 Summary

The analysis forecasts next-month stock excess returns using information available at each monthly decision date.

The target is constructed through a calendar-month stock join so that only the immediately following month's realised excess return is attached to the current forecast date.

The feature set uses 18 admissible numeric characteristics and FF49 industry membership. Identifiers and realised return outcomes are excluded.

Missing continuous characteristics are filled using contemporaneous industry-month medians, with same-month market medians used as fallback. Indicators identify observations that were originally missing.

Continuous variables are standardised using parameters estimated from the training sample. `down_market` and missingness indicators are retained as binary variables, while FF49 industry membership is one-hot encoded with one reference category omitted.

The prescribed chronological training, validation, and test periods are retained without modification. The resulting feature information will be used consistently for both pooled OLS and Gradient Boosting.
